# r10 step D: verifier, dev with and without re-reading vs r8p, suites, adopt or keep
About 4-6 h.

In [ ]:
import os, glob, json, hashlib, shutil, subprocess
W = '/kaggle/working'
PUB, PRIV = f'{W}/public', f'{W}/private'   # public: counts only, safe to commit; private: real question text, never commit
os.makedirs(PUB, exist_ok=True)
os.makedirs(PRIV, exist_ok=True)
REPO, BRANCH = '/kaggle/temp/Kinthread-llm-model', 'main'
R8P_SHA = '38c289d351d6517b5cde1bc9fb64ea97bd33eee193639d6351edda52fdfd7a52'


def sh(cmd):
    """Runs one shell command; stops the notebook if it fails."""
    print('$', cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f'STOP: command failed: {cmd}')


def stop(msg):
    raise SystemExit('STOP: ' + msg)


def par(*cmds):
    """Runs commands at once, one T4 each (one dev pass takes ~6 h; two in a row pass Kaggle's 12 h limit)."""
    procs = []
    for i, cmd in enumerate(cmds):
        print(f'$ [gpu {i % 2}]', cmd, flush=True)
        procs.append((cmd, subprocess.Popen(cmd, shell=True, env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(i % 2)})))
    bad = [cmd for cmd, p in procs if p.wait()]
    if bad:
        raise SystemExit(f'STOP: command failed: {bad[0]}')


def find(pattern):
    return sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))


def one(pattern, what):
    got = find(pattern)
    if not got:
        stop(f'attach {what} (no {pattern} under /kaggle/input)')
    return got[0]


def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(REPO):
    sh(f'git clone --depth 1 -b {BRANCH} https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}')
os.chdir(REPO)
tgz = find('real_questions_full.tgz')
if not tgz:
    stop('attach the private dataset kinthread-real-questions (it holds real_questions_full.tgz)')
sh(f'rm -rf data/real_questions && tar -xzf {tgz[0]}')
if not glob.glob('data/real_questions/*.jsonl'):
    stop('no real question files after unpacking real_questions_full.tgz')
print('repo and real questions ready')

R10 = one('runs/v10-r10/best.pt', 'the output of kinthread-r10-c-train')
A_DEV = one('private/r8p_dev_final.json', 'the output of kinthread-r9p-a-baseline')
B1 = json.load(open(one('public/B_r8p_summary.json', 'the output of kinthread-r9p-b1-suites-r8p')))
print('r10', sha(R10)[:12])

In [ ]:
sh(f'python train_verifier.py --ckpt {R10} --n 1500 --out {PUB}/r10_verifier.json')
BASE = f'--adaptive 1.0 --hybrid --verifier {PUB}/r10_verifier.json --cover --split_search --para 2 --k 16'
EXTRA_FLAGS = {'final': '', 'final_correct': '--correct'}
par(*[f'python -m eval.run_real --ckpt {R10} --split dev {BASE} {extra} --out {PRIV}/r10_dev_{name}.json'
      for name, extra in EXTRA_FLAGS.items()])
for name in EXTRA_FLAGS:
    sh(f'python -m eval.compare_runs {A_DEV} {PRIV}/r10_dev_{name}.json --out {PUB}/compare_{name}.json')

In [ ]:
par(f'python -m eval.gauntlet --ckpt {R10} --k 16 --out {PUB}/r10_gauntlet.json',
    f'python -m eval.consistency --groups 300 --ckpt {R10} --k 16 --out {PUB}/r10_consistency.json && '
    f'python -m eval.evaluate --ckpt {R10} --sets unseen --out {PUB}/r10_unseen.json')
g, c = json.load(open(f'{PUB}/r10_gauntlet.json')), json.load(open(f'{PUB}/r10_consistency.json'))
cmp = {n: json.load(open(f'{PUB}/compare_{n}.json')) for n in EXTRA_FLAGS}
passing = [n for n in cmp if cmp[n]['adopt']]   # a config meeting the rule beats a bigger non-significant gain
best = max(passing or cmp, key=lambda n: cmp[n]['gain_points'])
suites_ok = (g['correct'] >= B1['gauntlet_correct'] - 0.01 and c['all variants agree'] >= B1['consistency_all_agree'] - 0.01)
decision = {'step': 'D', 'adopt': bool(cmp[best]['adopt'] and suites_ok), 'config': best,
            'extra_flags': EXTRA_FLAGS[best], 'dev': cmp[best], 'all_configs': cmp,
            'gauntlet': {'r8p': B1['gauntlet_correct'], 'r10': g['correct']},
            'consistency_all_agree': {'r8p': B1['consistency_all_agree'], 'r10': c['all variants agree']},
            'suites_not_worse': suites_ok, 'r10_sha256': sha(R10),
            'rule': 'adopt = dev fair gain >= 2 points, wins > losses, McNemar p < 0.05, gauntlet and consistency not lower by more than 1 point'}
json.dump(decision, open(f'{PUB}/decision.json', 'w'), indent=1)
print(json.dumps(decision, indent=1))
print('D DONE: ADOPT r10' if decision['adopt'] else 'D DONE: KEEP r8p')